# 10 — Match and Season Impact of xG Models

This notebook aggregates shot-level xG predictions to match level, compares
model-implied xG outcomes against actual results, and builds a season-level
league table from xG-based expected points for the 2017/18 English Premier
League season.

**Data sources**
- Shot-level predictions from the baseline and combined xG models
- Official match scores from Wyscout `matches_England.json`
- Team names from Wyscout `teams.json`

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.append(os.path.abspath(".."))

from src.config import INTERIM_DIR, PROCESSED_DIR, FIGURES_DIR, MATCH_DIR, TEAM_FILE
from src.data_loader import load_matches, load_teams

In [ ]:
MATCH_JSON = MATCH_DIR / "matches_England.json"
TEAMS_JSON = TEAM_FILE
PRED_FILE  = INTERIM_DIR / "wyscout_test_xg_combined_predictions.parquet"

assert PRED_FILE.exists(), f"Missing: {PRED_FILE}"
assert MATCH_JSON.exists(), f"Missing: {MATCH_JSON}"
assert TEAMS_JSON.exists(), f"Missing: {TEAMS_JSON}"
print("Paths OK")

## Step 1: Load shot-level predictions

In [3]:
KEEP_COLS = ["matchId", "teamId", "is_goal", "xg_pred", "xg_pred_combined"]

preds = pd.read_parquet(PRED_FILE, columns=KEEP_COLS)
assert len(preds) == 8_881, f"Expected 8881 rows, got {len(preds)}"
assert preds[KEEP_COLS].notna().all().all(), "Unexpected nulls in predictions"

print(f"Predictions: {len(preds):,} shots")
preds.head()

Predictions: 8,881 shots


,matchId,teamId,is_goal,xg_pred,xg_pred_combined
0,2499719,1609,1,0.137206,0.122490
1,2499719,1631,0,0.117335,0.103708
2,2499719,1631,1,0.421288,0.397953
3,2499719,1609,0,0.044315,0.039344
4,2499719,1609,0,0.020479,0.015508


## Step 2: Aggregate shots to match-team level

In [4]:
shot_agg = (
    preds
    .groupby(["matchId", "teamId"])
    .agg(
        baseline_match_xg=("xg_pred", "sum"),
        combined_match_xg=("xg_pred_combined", "sum"),
        shot_count=("is_goal", "size"),
        shot_goals=("is_goal", "sum"),
    )
    .reset_index()
)

# 759 team-match pairs (Swansea has zero shots in match 2500013)
print(f"Shot aggregates: {len(shot_agg)} team-match pairs")
print(f"Unique matches:  {shot_agg['matchId'].nunique()}")
print(f"Unique teams:    {shot_agg['teamId'].nunique()}")
shot_agg.head()

Shot aggregates: 759 team-match pairs
Unique matches:  380
Unique teams:    20


,matchId,teamId,baseline_match_xg,combined_match_xg,shot_count,shot_goals
0,2499719,1609,2.155847,1.833557,27,4
1,2499719,1631,1.393207,1.334709,7,3
2,2499720,1625,1.303738,1.047193,14,1
3,2499720,1651,0.542681,0.442291,6,0
4,2499721,1610,1.330146,1.070922,16,2


## Step 3: Load official match scores from JSON

In [5]:
raw_matches = load_matches("England")

epl_matches = [m for m in raw_matches if m["seasonId"] == 181150]
assert len(epl_matches) == 380, f"Expected 380 EPL matches, got {len(epl_matches)}"

match_rows = []
for match in epl_matches:
    # Exactly two teams per match, one home and one away
    assert len(match["teamsData"]) == 2, (
        f"Match {match['wyId']}: expected 2 teams, got {len(match['teamsData'])}"
    )
    sides = sorted(td["side"] for td in match["teamsData"].values())
    assert sides == ["away", "home"], (
        f"Match {match['wyId']}: expected ['away','home'], got {sides}"
    )

    home_team_id = home_goals = away_team_id = away_goals = None
    for tid_str, td in match["teamsData"].items():
        assert int(tid_str) == td["teamId"], (
            f"Key/teamId mismatch: {tid_str} vs {td['teamId']}"
        )
        assert td["score"] >= 0, (
            f"Negative score in match {match['wyId']}, team {td['teamId']}"
        )
        if td["side"] == "home":
            home_team_id = td["teamId"]
            home_goals = td["score"]
        elif td["side"] == "away":
            away_team_id = td["teamId"]
            away_goals = td["score"]
        else:
            raise ValueError(f"Unexpected side: {td['side']}")

    match_rows.append({
        "matchId":      match["wyId"],
        "gameweek":     match["gameweek"],
        "match_label":  match["label"],
        "winner":       match["winner"],
        "home_team_id": home_team_id,
        "home_goals":   home_goals,
        "away_team_id": away_team_id,
        "away_goals":   away_goals,
    })

match_df: pd.DataFrame = pd.DataFrame(match_rows)
assert match_df["matchId"].nunique() == 380, "Duplicate matchIds"
print(f"Loaded {len(match_df)} EPL matches")
match_df.head()

Loaded 380 EPL matches


,matchId,gameweek,match_label,winner,home_team_id,home_goals,away_team_id,away_goals
0,2500089,38,"Burnley - AFC Bournemouth, 1 - 2",1659,1646,1,1659,2
1,2500090,38,"Crystal Palace - West Bromwich Albion, 2 - 0",1628,1628,2,1627,0
2,2500091,38,"Huddersfield Town - Arsenal, 0 - 1",1609,1673,0,1609,1
3,2500092,38,"Liverpool - Brighton & Hove Albion, 4 - 0",1612,1612,4,1651,0
4,2500093,38,"Manchester United - Watford, 1 - 0",1611,1611,1,1644,0


## Step 4: Build team-match scaffold and merge shot aggregates

In [ ]:
home_rows = match_df.rename(columns={
    "home_team_id": "teamId",
    "home_goals": "official_goals",
    "away_goals": "opponent_goals",
}).assign(is_home=1)
home_rows = home_rows.drop(columns=["away_team_id"])

away_rows = match_df.rename(columns={
    "away_team_id": "teamId",
    "away_goals": "official_goals",
    "home_goals": "opponent_goals",
}).assign(is_home=0)
away_rows = away_rows.drop(columns=["home_team_id"])

team_match = pd.concat([home_rows, away_rows], ignore_index=True)

# Fill zeros for teams with no shots (e.g. Swansea zero-shot match 2500013)
FILL_COLS = ["baseline_match_xg", "combined_match_xg", "shot_count", "shot_goals"]
team_match = team_match.merge(shot_agg, on=["matchId", "teamId"], how="left")
for col in FILL_COLS:
    team_match[col] = team_match[col].fillna(0)
team_match["shot_count"] = team_match["shot_count"].astype(int)
team_match["shot_goals"] = team_match["shot_goals"].astype(int)

assert len(team_match) == 760, f"Expected 760 team-match rows, got {len(team_match)}"
assert team_match[["matchId", "teamId"]].duplicated().sum() == 0, "Duplicate (matchId, teamId)"

zero_shot = team_match[team_match["shot_count"] == 0]
print(f"Zero-shot team-match pairs: {len(zero_shot)}")
if len(zero_shot) > 0:
    display(zero_shot)

print(f"\nTeam-match scaffold: {len(team_match)} rows")
team_match.head()

## Step 5: Load team names and validate team sets

In [7]:
teams_df   = load_teams()
team_names = dict(zip(teams_df["wyId"], teams_df["name"]))

# Team-set equality: predictions vs JSON matches
pred_teams = set(preds["teamId"].unique())
json_teams = set(team_match["teamId"].unique())
assert pred_teams == json_teams, (
    f"Team mismatch – in preds only: {pred_teams - json_teams}, "
    f"in JSON only: {json_teams - pred_teams}"
)
assert len(pred_teams) == 20, f"Expected 20 EPL teams, got {len(pred_teams)}"

# All EPL team IDs present in teams.json
missing = pred_teams - set(team_names.keys())
assert not missing, f"Teams missing from teams.json: {missing}"

team_match["team_name"] = team_match["teamId"].map(team_names)
assert team_match["team_name"].notna().all(), "Unmapped team IDs"

print(f"20 EPL teams mapped:")
print(sorted(team_match["team_name"].unique()))

20 EPL teams mapped:
['AFC Bournemouth', 'Arsenal', 'Brighton & Hove Albion', 'Burnley', 'Chelsea', 'Crystal Palace', 'Everton', 'Huddersfield Town', 'Leicester City', 'Liverpool', 'Manchester City', 'Manchester United', 'Newcastle United', 'Southampton', 'Stoke City', 'Swansea City', 'Tottenham Hotspur', 'Watford', 'West Bromwich Albion', 'West Ham United']


## Step 6: Shot-goals vs official-goals audit

Own goals are not recorded as shots in the Wyscout event data, so `shot_goals`
(summed from `is_goal`) can undercount official scores. Every mismatch should be
exactly −1, confirming the discrepancy is explained by own goals.

In [8]:
audit = team_match[["matchId", "teamId", "team_name", "official_goals", "shot_goals"]].copy()
mismatches = audit[audit["shot_goals"] != audit["official_goals"]]

assert len(mismatches) == 30, f"Expected 30 mismatches, got {len(mismatches)}"
assert (mismatches["shot_goals"] - mismatches["official_goals"] == -1).all(), (
    "All mismatches should be shot_goals undercounting by exactly 1"
)

print(f"Mismatches: {len(mismatches)} / {len(audit)} team-match pairs")
print("All mismatches are shot_goals undercounting by exactly 1 (own goals)")
display(mismatches.head(10))

Mismatches: 30 / 760 team-match pairs
All mismatches are shot_goals undercounting by exactly 1 (own goals)


,matchId,teamId,team_name,official_goals,shot_goals
8,2500097,1624,Tottenham Hotspur,5,4
60,2500041,1651,Brighton & Hove Albion,1,0
83,2500011,1610,Chelsea,2,1
84,2500012,1623,Everton,2,1
109,2499994,1631,Leicester City,1,0
132,2499968,1624,Tottenham Hotspur,2,1
141,2499955,1619,Southampton,1,0
164,2499934,1625,Manchester City,3,2
187,2499917,1644,Watford,2,1
228,2499877,1624,Tottenham Hotspur,5,4


## Step 7: Compute actual points and xG-implied points

In [ ]:
# --- Actual points from official scores (not from winner) ---
conditions = [
    team_match["official_goals"] > team_match["opponent_goals"],   # win
    team_match["official_goals"] == team_match["opponent_goals"],  # draw
    team_match["official_goals"] < team_match["opponent_goals"],   # loss
]
team_match["actual_points"] = np.select(conditions, [3, 1, 0])

# --- Winner field audit (non-blocking) ---
# The JSON winner field has known data errors; scores are authoritative.
winner_mismatches = []
for m in match_df.to_dict("records"):
    if m["home_goals"] > m["away_goals"]:
        expected_winner = m["home_team_id"]
    elif m["home_goals"] == m["away_goals"]:
        expected_winner = 0
    else:
        expected_winner = m["away_team_id"]
    if m["winner"] != expected_winner:
        winner_mismatches.append(m)

assert len(winner_mismatches) == 2, (
    f"Expected 2 winner-field mismatches, got {len(winner_mismatches)}"
)
print(f"Winner field audit: {len(winner_mismatches)} known data errors in Wyscout JSON")
for m in winner_mismatches:
    print(f"  Match {m['matchId']}: {m['match_label']} — "
          f"winner={m['winner']} (should be non-zero, score {m['home_goals']}-{m['away_goals']})")

# --- xG-implied points ---
# Need opponent xG for comparison; merge opponent's xG onto each row
opponent_xg = team_match[["matchId", "teamId", "baseline_match_xg", "combined_match_xg"]].rename(
    columns={
        "teamId": "opponent_id",
        "baseline_match_xg": "opp_baseline_xg",
        "combined_match_xg": "opp_combined_xg",
    }
)

# Each match has exactly 2 teams; merge via matchId, excluding self
team_match = team_match.merge(
    opponent_xg,
    left_on="matchId",
    right_on="matchId",
)
# Keep only rows where opponent_id != teamId (the actual opponent)
team_match = team_match[team_match["teamId"] != team_match["opponent_id"]].copy()

assert len(team_match) == 760, f"Expected 760 after opponent merge, got {len(team_match)}"

conditions_bl = [
    team_match["baseline_match_xg"] > team_match["opp_baseline_xg"],
    team_match["baseline_match_xg"] == team_match["opp_baseline_xg"],
    team_match["baseline_match_xg"] < team_match["opp_baseline_xg"],
]
team_match["baseline_xg_points"] = np.select(conditions_bl, [3, 1, 0])

conditions_cb = [
    team_match["combined_match_xg"] > team_match["opp_combined_xg"],
    team_match["combined_match_xg"] == team_match["opp_combined_xg"],
    team_match["combined_match_xg"] < team_match["opp_combined_xg"],
]
team_match["combined_xg_points"] = np.select(conditions_cb, [3, 1, 0])

print(f"Points computed for {len(team_match)} team-match rows")
team_match[["team_name", "matchId", "official_goals", "opponent_goals",
            "actual_points", "combined_match_xg", "opp_combined_xg",
            "combined_xg_points"]].head(10)

## Step 8: Season league table — actual vs xG

In [10]:
season = (
    team_match
    .groupby(["teamId", "team_name"])
    .agg(
        played=("matchId", "size"),
        actual_pts=("actual_points", "sum"),
        goals_for=("official_goals", "sum"),
        goals_against=("opponent_goals", "sum"),
        xg_for_baseline=("baseline_match_xg", "sum"),
        xg_against_baseline=("opp_baseline_xg", "sum"),
        baseline_xg_pts=("baseline_xg_points", "sum"),
        xg_for_combined=("combined_match_xg", "sum"),
        xg_against_combined=("opp_combined_xg", "sum"),
        combined_xg_pts=("combined_xg_points", "sum"),
    )
    .reset_index()
)

season["goal_diff"] = season["goals_for"] - season["goals_against"]
season["xg_diff_baseline"] = season["xg_for_baseline"] - season["xg_against_baseline"]
season["xg_diff_combined"] = season["xg_for_combined"] - season["xg_against_combined"]

# Sort by actual points (desc), then goal difference (desc)
season = season.sort_values(
    ["actual_pts", "goal_diff", "goals_for"],
    ascending=[False, False, False],
    ignore_index=True,
)
season.index = season.index + 1  # 1-based rank

assert len(season) == 20, f"Expected 20 teams, got {len(season)}"
assert (season["played"] == 38).all(), "Not all teams played 38 matches"

display_cols = [
    "team_name", "played", "actual_pts", "goal_diff",
    "goals_for", "goals_against",
    "combined_xg_pts", "xg_diff_combined",
    "baseline_xg_pts", "xg_diff_baseline",
]
season_display: pd.DataFrame = season[display_cols].copy()
season_display.columns = [
    "Team", "P", "Pts", "GD", "GF", "GA",
    "xG Pts (comb)", "xGD (comb)",
    "xG Pts (base)", "xGD (base)",
]
for col in ["xGD (comb)", "xGD (base)"]:
    season_display[col] = season_display[col].round(1)

print("2017/18 EPL Season Table — Actual vs xG")
display(season_display)

2017/18 EPL Season Table — Actual vs xG


,Team,P,Pts,GD,GF,GA,xG Pts (comb),xGD (comb),xG Pts (base),xGD (base)
1,Manchester City,38,100,79,106,27,108,61.8,108,55.9
2,Manchester United,38,81,40,68,28,75,17.3,66,14.7
3,Tottenham Hotspur,38,77,38,74,36,90,33.6,87,27.5
4,Liverpool,38,75,46,84,38,99,42.7,99,35.3
5,Chelsea,38,70,24,62,38,63,20.8,66,19.7
6,Arsenal,38,63,23,74,51,72,20.7,66,20.7
7,Burnley,38,54,-3,36,39,48,-17.3,42,-16.3
8,Everton,38,49,-14,44,58,39,-12.7,42,-11.9
9,Leicester City,38,47,-4,56,60,69,-3.9,66,-3.8
10,Newcastle United,38,44,-8,39,47,48,-14.6,48,-12.1


## Step 9: Actual vs xG points scatter plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(season["actual_pts"], season["combined_xg_pts"], s=60, zorder=3)

# 45-degree reference line
lo = min(season["actual_pts"].min(), season["combined_xg_pts"].min()) - 3
hi = max(season["actual_pts"].max(), season["combined_xg_pts"].max()) + 3
ax.plot([lo, hi], [lo, hi], ls="--", color="grey", alpha=0.6, zorder=1)

# Label each point with team name
for _, row in season.iterrows():
    ax.annotate(
        row["team_name"],
        (row["actual_pts"], row["combined_xg_pts"]),
        fontsize=7, ha="left", va="bottom",
        xytext=(4, 4), textcoords="offset points",
    )

ax.set_xlabel("Actual Points")
ax.set_ylabel("xG Points (Combined Model)")
ax.set_title("2017/18 EPL: Actual vs xG-Implied Season Points")
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)

fig.savefig(
    str(FIGURES_DIR / "wyscout_xg_season_points_comparison.png"),
    dpi=150, bbox_inches="tight",
)
plt.close(fig)
print("Saved: wyscout_xg_season_points_comparison.png")

## Step 10: Biggest match-level xG surprises

In [ ]:
match_level = (
    match_df[["matchId", "gameweek", "match_label", "home_team_id", "home_goals", "away_team_id", "away_goals"]]
    .copy()
)

home_xg = team_match.loc[
    team_match["is_home"] == 1,
    ["matchId", "combined_match_xg", "opp_combined_xg"],
].rename(columns={
    "combined_match_xg": "home_xg",
    "opp_combined_xg": "away_xg",
})
match_level = match_level.merge(home_xg, on="matchId", validate="one_to_one")

match_level["actual_gd"] = match_level["home_goals"] - match_level["away_goals"]
match_level["xg_gd"] = match_level["home_xg"] - match_level["away_xg"]
match_level["xg_surprise"] = match_level["actual_gd"] - match_level["xg_gd"]

surprise_cols = ["gameweek", "match_label", "home_goals", "away_goals",
                 "home_xg", "away_xg", "actual_gd", "xg_gd", "xg_surprise"]
for col in ["home_xg", "away_xg", "xg_gd", "xg_surprise"]:
    match_level[col] = match_level[col].round(2)

print("Top 10 positive xG surprises (home team outperformed xG):")
display(
    match_level
    .nlargest(10, "xg_surprise")[surprise_cols]
    .reset_index(drop=True)
)

print("\nTop 10 negative xG surprises (home team underperformed xG):")
display(
    match_level
    .nsmallest(10, "xg_surprise")[surprise_cols]
    .reset_index(drop=True)
)

## Step 11: Save outputs

In [ ]:
season.to_csv(PROCESSED_DIR / "wyscout_xg_season_table.csv", index=True)
print(f"Saved season table: wyscout_xg_season_table.csv ({len(season)} teams)")

match_level.to_parquet(PROCESSED_DIR / "wyscout_xg_match_level.parquet", index=False)
print(f"Saved match-level: wyscout_xg_match_level.parquet ({len(match_level)} matches)")

team_match.to_parquet(PROCESSED_DIR / "wyscout_xg_team_match.parquet", index=False)
print(f"Saved team-match:  wyscout_xg_team_match.parquet ({len(team_match)} rows)")

check_season = pd.read_csv(PROCESSED_DIR / "wyscout_xg_season_table.csv", index_col=0)
assert check_season.shape[0] == 20, f"Season table reload: expected 20, got {check_season.shape[0]}"

check_match = pd.read_parquet(PROCESSED_DIR / "wyscout_xg_match_level.parquet")
assert check_match.shape[0] == 380, f"Match-level reload: expected 380, got {check_match.shape[0]}"

check_tm = pd.read_parquet(PROCESSED_DIR / "wyscout_xg_team_match.parquet")
assert check_tm.shape[0] == 760, f"Team-match reload: expected 760, got {check_tm.shape[0]}"

print("\nAll outputs saved and verified")